In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models, transforms
from PIL import Image
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings

# --- 1. Configuration ---

# Suppress warnings
warnings.filterwarnings("ignore")

# Environment settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [10]:
import os
import shutil

# Path to Kaggle working directory
work_dir = "/kaggle/working"

# Option 1: Remove all files and folders in working directory
for item in os.listdir(work_dir):
    item_path = os.path.join(work_dir, item)
    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.unlink(item_path)  # Remove file or symlink
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)  # Remove directory
        print(f"Deleted: {item_path}")
    except Exception as e:
        print(f"Failed to delete {item_path}. Reason: {e}")

# Option 2: Remove only specific file types (e.g., CSVs)
# for item in os.listdir(work_dir):
#     if item.endswith(".csv"):
#         os.remove(os.path.join(work_dir, item))
#         print(f"Deleted CSV: {item}")


In [22]:
# File paths
LABELED_DIR = "/kaggle/input/autistic-children-emotions-dr-fatma-m-talaat/Autism emotion recogition dataset/Autism emotion recogition dataset/train"
UNLABELED_DIR = "/kaggle/input/fl-fer-rawimages/Concatenated"
WORKING_DIR = "/kaggle/working/"
OUTPUT_CSV = os.path.join(WORKING_DIR, "pseudo_labels_aug.csv")

# Model & Training parameters
NUM_ITERATIONS = 5
NUM_EPOCHS_PER_ITERATION = 10  
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

In [23]:
# Dynamically find emotion classes from the directory structure
try:
    EMOTION_CLASSES = [d for d in os.listdir(LABELED_DIR) if os.path.isdir(os.path.join(LABELED_DIR, d))]
    if not EMOTION_CLASSES:
        raise FileNotFoundError
    print(f"Found {len(EMOTION_CLASSES)} emotion classes: {EMOTION_CLASSES}")
except FileNotFoundError:
    print(f"Error: Could not find labeled data directory or no subdirectories found in {LABELED_DIR}")
    print("Please double-check the path.")
    EMOTION_CLASSES = [] # Stop script if path is wrong

Found 6 emotion classes: ['joy', 'surprise', 'fear', 'sadness', 'Natural', 'anger']


In [24]:
# Image transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [25]:
def create_model():
    """Creates a pre-trained ResNet-18 model for binary classification."""
    model = models.resnet18(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(DEVICE)

# --- 3. Custom Datasets ---

class OneVsAllDataset(Dataset):
    """
    Creates a binary (one-vs-all) dataset from the original labeled data.
    Label 1 for `target_class`, Label 0 for all other classes.
    """
    def __init__(self, base_dir, all_classes, target_class, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for cls in all_classes:
            class_dir = os.path.join(base_dir, cls)
            label = 1 if cls == target_class else 0
            
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                self.image_paths.append(img_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Warning: Could not load image {img_path}. Skipping. Error: {e}")
            # Return a dummy image and label if load fails
            return torch.randn(3, 224, 224), torch.tensor(0.0)

        return image, torch.tensor(label, dtype=torch.float)

class PseudoLabeledDataset(Dataset):
    """
    Creates a binary dataset from a list of pseudo-labeled images.
    `all_pseudo_data` is a list of tuples: [(img_path, assigned_class), ...]
    """
    def __init__(self, all_pseudo_data, target_class, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for img_path, assigned_class in all_pseudo_data:
            label = 1 if assigned_class == target_class else 0
            self.image_paths.append(img_path)
            self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Warning: Could not load pseudo-image {img_path}. Skipping. Error: {e}")
            return torch.randn(3, 224, 224), torch.tensor(0.0)

        return image, torch.tensor(label, dtype=torch.float)

class UnlabeledDataset(Dataset):
    """
    Dataset for unlabeled images. Returns the image and its path for tracking.
    """
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Warning: Could not load unlabeled image {img_path}. Skipping. Error: {e}")
            # Return a dummy image and the path
            return torch.randn(3, 224, 224), img_path

        return image, img_path

# --- 4. Helper Function ---

def train_model(model, loader):
    """Utility function to train a model for one epoch."""
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    for _ in range(NUM_EPOCHS_PER_ITERATION):
        epoch_loss = 0
        for inputs, labels in tqdm(loader, desc="Training"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Epoch Loss: {epoch_loss/len(loader):.4f}")
    return model

# --- 5. Main Iterative Logic ---

def main():
    if not EMOTION_CLASSES:
        print("No emotion classes found. Exiting.")
        return

    # Initialize tracking of unlabeled images
    try:
        unlabeled_image_files = [
            os.path.join(UNLABELED_DIR, f) 
            for f in os.listdir(UNLABELED_DIR) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        if not unlabeled_image_files:
            print(f"No images found in {UNLABELED_DIR}. Exiting.")
            return
            
        print(f"Found {len(unlabeled_image_files)} unlabeled images.")
    except FileNotFoundError:
        print(f"Error: Unlabeled data directory not found at {UNLABELED_DIR}")
        return

    df_unlabeled = pd.DataFrame(unlabeled_image_files, columns=['path'])
    df_unlabeled['assigned_class'] = np.nan
    df_unlabeled['iteration_labeled'] = np.nan
    
    all_pseudo_labeled_data = [] # Stores (path, assigned_class) tuples

    for i in range(NUM_ITERATIONS):
        print("\n" + "="*50)
        print(f"--- Starting Iteration {i+1} / {NUM_ITERATIONS} ---")
        print(f"Current pseudo-labels in pool: {len(all_pseudo_labeled_data)}")

        # Get images that are still unlabeled
        current_unlabeled_paths = df_unlabeled[df_unlabeled['assigned_class'].isna()]['path'].tolist()
        
        if not current_unlabeled_paths:
            print("All images have been pseudo-labeled. Stopping early.")
            break

        print(f"Training on {len(current_unlabeled_paths)} remaining unlabeled images.")
        
        models_dict = {}
        all_predictions = {} # To store predictions: {path: {emotion: prob}}

        # --- A. Training Phase ---
        for emotion in EMOTION_CLASSES:
            print(f"\nTraining model for: {emotion}")
            
            # 1. Original labeled data
            original_dataset = OneVsAllDataset(LABELED_DIR, EMOTION_CLASSES, emotion, transform=data_transforms)
            
            # 2. Newly found pseudo-labeled data (from previous iterations)
            pseudo_dataset = PseudoLabeledDataset(all_pseudo_labeled_data, emotion, transform=data_transforms)
            
            # 3. Combine them
            combined_dataset = ConcatDataset([original_dataset, pseudo_dataset])
            train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
            
            print(f"Training '{emotion}' model with {len(combined_dataset)} images ({len(original_dataset)} original + {len(pseudo_dataset)} pseudo)")
            
            model = create_model()
            model = train_model(model, train_loader)
            models_dict[emotion] = model

        # --- B. Prediction Phase ---
        print("\nPredicting on remaining unlabeled data...")
        unlabeled_dataset = UnlabeledDataset(current_unlabeled_paths, transform=data_transforms)
        unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        
        # Initialize prediction storage
        predictions_df = pd.DataFrame(index=current_unlabeled_paths)

        for emotion, model in models_dict.items():
            model.eval()
            probas = []
            paths = []
            
            with torch.no_grad():
                for inputs, img_paths in tqdm(unlabeled_loader, desc=f"Predicting {emotion}"):
                    inputs = inputs.to(DEVICE)
                    logits = model(inputs).squeeze(1)
                    probabilities = torch.sigmoid(logits)
                    
                    probas.extend(probabilities.cpu().numpy())
                    paths.extend(img_paths)
            
            # Store probabilities for this emotion
            temp_df = pd.DataFrame({'path': paths, emotion: probas})
            temp_df = temp_df.set_index('path')
            predictions_df = predictions_df.join(temp_df)

        # --- C. Pseudo-Labeling Phase ---
        print("\nAssigning pseudo-labels based on your logic...")
        
        # Convert probabilities to binary predictions (1 if > 0.5, else 0)
        binary_predictions = (predictions_df > 0.5).astype(int)
        
        # Count how many models predicted '1' (positive)
        binary_predictions['positive_counts'] = binary_predictions.sum(axis=1)
        
        # Find images with *exactly one* positive prediction
        confident_predictions = binary_predictions[binary_predictions['positive_counts'] == 1]
        
        newly_labeled_count = 0
        for path, row in confident_predictions.iterrows():
            # Find which column (emotion) was '1'
            assigned_class = row.drop('positive_counts').idxmax()
            
            # Add to the global pool for the *next* iteration
            all_pseudo_labeled_data.append((path, assigned_class))
            
            # Update our master dataframe for tracking
            df_unlabeled.loc[df_unlabeled['path'] == path, 'assigned_class'] = assigned_class
            df_unlabeled.loc[df_unlabeled['path'] == path, 'iteration_labeled'] = i + 1
            
            newly_labeled_count += 1
            
        print(f"--- Iteration {i+1} Summary ---")
        print(f"Found {newly_labeled_count} new confident labels.")
        
        if newly_labeled_count == 0:
            print("No new confident labels found in this iteration. Stopping.")
            break

    # --- 6. Final Report ---
    print("\n" + "="*50)
    print("Iterative process complete.")
    
    # Save the final dataframe to CSV
    df_unlabeled.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved to {OUTPUT_CSV}")
    
    print("\nFinal Pseudo-Label Counts (including unassigned):")
    print(df_unlabeled['assigned_class'].value_counts(dropna=False))

In [26]:
if __name__ == "__main__":
    main()

Found 3569 unlabeled images.

--- Starting Iteration 1 / 5 ---
Current pseudo-labels in pool: 0
Training on 3569 remaining unlabeled images.

Training model for: joy
Training 'joy' model with 1199 images (1199 original + 0 pseudo)


Training: 100%|██████████| 38/38 [00:07<00:00,  5.02it/s]


Epoch Loss: 0.6190


Training: 100%|██████████| 38/38 [00:04<00:00,  7.93it/s]


Epoch Loss: 0.1146


...

Epoch Loss: 0.0134


Training: 100%|██████████| 141/141 [00:16<00:00,  8.74it/s]


Epoch Loss: 0.0138

Predicting on remaining unlabeled data...


Predicting anger: 100%|██████████| 9/9 [00:01<00:00,  8.08it/s]



Assigning pseudo-labels based on your logic...
--- Iteration 5 Summary ---
Found 143 new confident labels.

Iterative process complete.
Results saved to /kaggle/working/pseudo_labels_aug.csv

Final Pseudo-Label Counts (including unassigned):
assigned_class
joy         2024
Natural      472
anger        382
sadness      254
surprise     203
NaN          125
fear         109
Name: count, dtype: int64


In [28]:
import pandas as pd
import os
import shutil
from tqdm import tqdm

# --- Configuration ---

# The CSV file generated by your previous script
INPUT_CSV = "/kaggle/working/pseudo_labels_aug.csv"

# The name of the temporary directory to build our dataset
TEMP_DIR = "/kaggle/working/pseudo_dataset_by_class"

# The final, desired name of the output zip file
OUTPUT_ZIP_FILE = "/kaggle/working/pseudo_labeled_dataset_aug"

print(f"Reading pseudo-labels from: {INPUT_CSV}")

# --- 1. Load the CSV File ---
try:
    df = pd.read_csv(INPUT_CSV)
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_CSV}")
    print("Please make sure your first script has run and created this file.")
    # Stop the script if the file doesn't exist
    raise

# --- 2. Filter for Successfully Labeled Images ---
labeled_df = df.dropna(subset=['assigned_class'])

if labeled_df.empty:
    print("No pseudo-labels were found in the CSV file. Nothing to zip.")
else:
    print(f"Found {len(labeled_df)} images with new pseudo-labels.")

    # --- 3. Create Temporary Directory Structure ---
    # Clean up old directory if it exists
    if os.path.exists(TEMP_DIR):
        print(f"Removing old temporary directory: {TEMP_DIR}")
        shutil.rmtree(TEMP_DIR)
    
    print(f"Creating new temporary directory: {TEMP_DIR}")
    os.makedirs(TEMP_DIR, exist_ok=True)
    
    # Get all unique new classes and create folders for them
    emotion_classes = labeled_df['assigned_class'].unique()
    for emotion in emotion_classes:
        class_dir = os.path.join(TEMP_DIR, str(emotion))
        os.makedirs(class_dir, exist_ok=True)
    
    print(f"Created {len(emotion_classes)} emotion subfolders.")

    # --- 4. Copy Images into New Folders ---
    print("Copying images into their new class folders...")
    
    copied_count = 0
    skipped_count = 0
    
    for _, row in tqdm(labeled_df.iterrows(), total=len(labeled_df)):
        original_path = row['path']
        assigned_class = row['assigned_class']
        
        # Get just the filename (e.g., 'image_123.png')
        filename = os.path.basename(original_path)
        
        # Define the new destination path
        destination_dir = os.path.join(TEMP_DIR, assigned_class)
        destination_path = os.path.join(destination_dir, filename)
        
        # Copy the file
        try:
            if os.path.exists(original_path):
                shutil.copy(original_path, destination_path)
                copied_count += 1
            else:
                print(f"\nWarning: Source file not found, skipping: {original_path}")
                skipped_count += 1
        except Exception as e:
            print(f"\nError copying {original_path}: {e}")
            skipped_count += 1

    print(f"Successfully copied {copied_count} images.")
    if skipped_count > 0:
        print(f"Skipped {skipped_count} images (see warnings above).")

    # --- 5. Create the Zip File ---
    print(f"\nCreating zip file: {OUTPUT_ZIP_FILE}.zip")
    
    try:
        shutil.make_archive(
            base_name=OUTPUT_ZIP_FILE,  # The path and name of the file to create (without .zip)
            format='zip',                # The format to use
            root_dir=TEMP_DIR          # The directory to zip up
        )
        print("Zip file created successfully!")
    except Exception as e:
        print(f"Error creating zip file: {e}")

    # # --- 6. (Optional) Clean up the Temporary Directory ---
    # try:
    #     print(f"Cleaning up temporary directory: {TEMP_DIR}")
    #     shutil.rmtree(TEMP_DIR)
    #     print("Cleanup complete.")
    # except Exception as e:
    #     print(f"Error cleaning up temp directory: {e}")

    print(f"\n--- Process Finished ---")
    print(f"Your new dataset is ready at: {OUTPUT_ZIP_FILE}.zip")

Reading pseudo-labels from: /kaggle/working/pseudo_labels_aug.csv
Found 3444 images with new pseudo-labels.
Removing old temporary directory: /kaggle/working/pseudo_dataset_by_class
Creating new temporary directory: /kaggle/working/pseudo_dataset_by_class
Created 6 emotion subfolders.
Copying images into their new class folders...


100%|██████████| 3444/3444 [00:05<00:00, 651.73it/s]


Successfully copied 3444 images.

Creating zip file: /kaggle/working/pseudo_labeled_dataset_aug.zip
Zip file created successfully!

--- Process Finished ---
Your new dataset is ready at: /kaggle/working/pseudo_labeled_dataset_aug.zip
